In [31]:
# Step 1: compute the equi-width bucketization bias for German Credit Dataset
import numpy as np
import pandas as pd
from MyData import load_xg_dataset
from MeasureBias import equibucket_bias

df = load_xg_dataset('datasets/german_credit_data.csv', x_idx=1, g_idx=2)
k = 6

print(equibucket_bias(df, k=k))

(np.float64(0.25024096385542166), array([0.25024096, 0.25024096]))


/Users/adam/Git/UnbiasedBinningPrivate/Python/MyData.py:75: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  g = g_raw.replace(g_mapping).values


In [32]:
# Step 2: Add the equi-width bucketed columns to the German Credit Dataset

import numpy as np
import pandas as pd

filepath='datasets/german_credit_data.csv'

Cols = ['Age', 'Credit amount', 'Duration']
df = pd.read_csv(filepath, delimiter=',')
n = df.shape[0]
K=[6,6,6] # number of buckets for each attribute
# Equi-width bucketization bias for these values of K: 
#                                                        .25, .093, .1

for j in range(len(Cols)):
    col = Cols[j]
    newcol = np.zeros(n,dtype=int)
    x = int(n/K[j]) # the bucket size
    for i in range(n): newcol[i] = int(i/x)
    df = df.sort_values(by=col)
    df[col+'_bucket'] = newcol
# save the new dataset
df.to_csv('datasets/german_credit_data_equibuckets.csv')


In [1]:
# Step 3: Train the models
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score



feature_cols = ['Age_bucket', 'Credit amount_bucket', 'Duration_bucket']
target_col= 'Risk'  # the target variable
df = pd.read_csv('datasets/german_credit_data_equibuckets.csv', delimiter=',')

X = df[feature_cols]
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y  # helps preserve class balance
)

preprocess = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), feature_cols)
    ]
)

log_reg_clf = Pipeline(steps=[
    ('preprocess', preprocess),
    ('model', LogisticRegression(max_iter=1000, solver='liblinear'))
])


svm_clf = Pipeline(steps=[
    ('preprocess', preprocess),
    ('model', LinearSVC())
])

log_reg_clf.fit(X_train, y_train)
svm_clf.fit(X_train, y_train)

y_pred_log = log_reg_clf.predict(X_test)
y_pred_svm = svm_clf.predict(X_test)

def evaluate_model(name, y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, pos_label='bad')
    print(f"{name}")
    print(f"  Accuracy: {acc:.4f}")
    print(f"  F1-score: {f1:.4f}")
    print("-" * 40)

evaluate_model("Logistic Regression", y_test, y_pred_log)
evaluate_model("Linear SVM", y_test, y_pred_svm)




Logistic Regression
  Accuracy: 0.6733
  F1-score: 0.2576
----------------------------------------
Linear SVM
  Accuracy: 0.6633
  F1-score: 0.2171
----------------------------------------


In [2]:
# =========================================
# STEP 4: Fairness evaluation with AIF360
# Sensitive attribute: Sex
# =========================================

import numpy as np
import pandas as pd
from aif360.datasets import BinaryLabelDataset
from aif360.metrics import ClassificationMetric

# ------------------------------------------------
# 1. Select the test subset and encode labels/sex
# ------------------------------------------------

# Indices of the test set from Step 1
test_index = X_test.index

# Subset of the original df corresponding to the test set
test_df = df.loc[test_index, ['Sex', 'Risk']].copy()

# Map labels: 'good'/'bad' -> 0/1  (0 = good, 1 = bad)
label_map = {'good': 0, 'bad': 1}
test_df['Risk_num'] = test_df['Risk'].map(label_map)

# Map Sex to numeric (adjust these if your values differ)
# For example, if your data uses 'male'/'female':
sex_map = {'male': 1, 'female': 0}
test_df['Sex_num'] = test_df['Sex'].map(sex_map)

# Safety check: if mapping fails, inspect unique values
if test_df['Sex_num'].isna().any():
    print("WARNING: Some Sex values were not mapped. Check sex_map and df['Sex'].unique().")
    print("Unique Sex values:", test_df['Sex'].unique())

# Encode predictions to numeric using the same label_map
y_true_num = test_df['Risk_num'].values
y_pred_log_num = pd.Series(y_pred_log, index=test_index).map(label_map).values
y_pred_svm_num = pd.Series(y_pred_svm, index=test_index).map(label_map).values

# ------------------------------------------------
# 2. Create BinaryLabelDataset objects
# ------------------------------------------------

# Features for fairness evaluation (only Sex_num is needed here)
X_fair = test_df[['Sex_num']].values

# True dataset
dataset_true = BinaryLabelDataset(
    favorable_label=0,          # 0 = 'good'
    unfavorable_label=1,        # 1 = 'bad'
    df=pd.DataFrame(
        data=np.hstack([X_fair, y_true_num.reshape(-1, 1)]),
        columns=['Sex_num', 'Risk_num']
    ),
    label_names=['Risk_num'],
    protected_attribute_names=['Sex_num']
)

# Logistic regression predictions
dataset_pred_log = dataset_true.copy()
dataset_pred_log.labels = y_pred_log_num.reshape(-1, 1)

# Decision tree predictions
dataset_pred_svm = dataset_true.copy()
dataset_pred_svm.labels = y_pred_svm_num.reshape(-1, 1)

# ------------------------------------------------
# 3. Define privileged / unprivileged groups
# ------------------------------------------------

# Privileged: Sex_num = 1 (male), Unprivileged: Sex_num = 0 (female)
# Change these if you choose a different mapping above.
privileged_groups = [{'Sex_num': 1}]
unprivileged_groups = [{'Sex_num': 0}]

# ------------------------------------------------
# 4. Helper function to compute fairness metrics
# ------------------------------------------------

def print_fairness_metrics(model_name, dataset_true, dataset_pred):
    metric = ClassificationMetric(
        dataset_true,
        dataset_pred,
        unprivileged_groups=unprivileged_groups,
        privileged_groups=privileged_groups
    )

    print(f"\n=== Fairness metrics for {model_name} ===")
    # Demographic / statistical parity
    spd = metric.statistical_parity_difference()
    di = metric.disparate_impact()

    # Equality of opportunity (TPR difference)
    eod = metric.equal_opportunity_difference()

    # Average odds difference (avg of FPR and TPR diffs)
    aod = metric.average_odds_difference()

    # Inequality measure
    theil = metric.theil_index()

    print(f"Statistical Parity Difference (SPD): {spd:.4f}")
    print(f"Disparate Impact (DI):              {di:.4f}")
    print(f"Equal Opportunity Difference:       {eod:.4f}")
    print(f"Average Odds Difference:            {aod:.4f}")
    print(f"Theil Index:                        {theil:.4f}")
    print("=========================================")

# ------------------------------------------------
# 5. Evaluate both models
# ------------------------------------------------

print_fairness_metrics("Logistic Regression", dataset_true, dataset_pred_log)
print_fairness_metrics("Linear SVM", dataset_true, dataset_pred_svm)


pip install 'aif360[AdversarialDebiasing]'
pip install 'aif360[AdversarialDebiasing]'
pip install 'aif360[Reductions]'
pip install 'aif360[Reductions]'
pip install 'aif360[inFairness]'
pip install 'aif360[Reductions]'



=== Fairness metrics for Logistic Regression ===
Statistical Parity Difference (SPD): -0.0545
Disparate Impact (DI):              0.9378
Equal Opportunity Difference:       -0.0794
Average Odds Difference:            -0.0341
Theil Index:                        0.1424

=== Fairness metrics for Linear SVM ===
Statistical Parity Difference (SPD): -0.0386
Disparate Impact (DI):              0.9563
Equal Opportunity Difference:       -0.0794
Average Odds Difference:            -0.0126
Theil Index:                        0.1432


In [ ]:
# Part 2, Step 1: decide what value of epsilon to use for each attribute

import numpy as np
from MyData import load_xg_dataset
from EbiasedBinning import ebias_binning

x_index = [1,7,8] 
k=6
e = [0.1,0.04,0.05]

boundary_indices = []

for i in range(len(e)):
    D = load_xg_dataset('datasets/german_credit_data.csv', x_idx=x_index[i], g_idx=2) # index 2 is Sex
    n = D.shape[0]
    result = ebias_binning(D,k=k,eps=e[i]) # the indices are in result[1]
    boundary_indices.append([result[1]])

print("The boundary indices for each attribute are:")
print(boundary_indices)


/Users/adam/Git/UnbiasedBinningPrivate/Python/MyData.py:75: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  g = g_raw.replace(g_mapping).values
/Users/adam/Git/UnbiasedBinningPrivate/Python/MyData.py:75: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  g = g_raw.replace(g_mapping).values
/Users/adam/Git/UnbiasedBinningPrivate/Python/MyData.py:75: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-i

The boundary indices for each attribute are:
[[[np.int64(165), np.int64(331), np.int64(498), np.int64(665), np.int64(832), 999]], [[np.int64(165), np.int64(331), np.int64(498), np.int64(665), np.int64(832), 999]], [[np.int64(165), np.int64(331), np.int64(498), np.int64(665), np.int64(832), 999]]]


### Result Log
The boundary indices for each attribute are:
[[[np.int64(165), np.int64(331), np.int64(498), np.int64(665), np.int64(832), 999]], [[np.int64(165), np.int64(331), np.int64(498), np.int64(665), np.int64(832), 999]], [[np.int64(165), np.int64(331), np.int64(498), np.int64(665), np.int64(832), 999]]]

In [28]:
# Part 2, Step 2: create the bucketized dataset using the e-biased binning
# you should first run the previous code cell to get 'boundary_indices'

import numpy as np
import pandas as pd

# now load the original dataset again to add the new column

filepath='datasets/german_credit_data.csv'

Cols = ['Age', 'Credit amount', 'Duration']
df = pd.read_csv(filepath, delimiter=',')
n = df.shape[0]
K=[6,6,6]


for j in range(len(Cols)):
    col = Cols[j]
    b = boundary_indices[j][0]
    newcol = np.zeros(n,dtype=int)
    l = 0
    for i in range(n): 
        newcol[i] = l
        if i== b[l]: l+=1
    df = df.sort_values(by=col)
    df[col+'_bucket_u'] = newcol


# save the new dataset
df.to_csv('datasets/german_credit_data_ebiased.csv')

Next Step: train the model using the unbiased binning and evaluate the fairness

In [3]:
# Part 2, Step 3: Train the models
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score



feature_cols = ['Age_bucket_u', 'Credit amount_bucket_u', 'Duration_bucket_u']
target_col= 'Risk'  # the target variable
df = pd.read_csv('datasets/german_credit_data_ebiased.csv', delimiter=',')

X = df[feature_cols]
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y  # helps preserve class balance
)

preprocess = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), feature_cols)
    ]
)

log_reg_clf = Pipeline(steps=[
    ('preprocess', preprocess),
    ('model', LogisticRegression(max_iter=1000, solver='liblinear'))
])

svm_clf = Pipeline(steps=[
    ('preprocess', preprocess),
    ('model', LinearSVC())
])


log_reg_clf.fit(X_train, y_train)
svm_clf.fit(X_train, y_train)

y_pred_log = log_reg_clf.predict(X_test)
y_pred_svm = svm_clf.predict(X_test)

def evaluate_model(name, y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, pos_label='bad')
    print(f"{name}")
    print(f"  Accuracy: {acc:.4f}")
    print(f"  F1-score: {f1:.4f}")
    print("-" * 40)

evaluate_model("Logistic Regression", y_test, y_pred_log)
evaluate_model("Linear SVM", y_test, y_pred_svm)

Logistic Regression
  Accuracy: 0.7033
  F1-score: 0.1835
----------------------------------------
Linear SVM
  Accuracy: 0.7033
  F1-score: 0.1835
----------------------------------------


In [4]:
# =========================================
# Part 2, STEP 4: Fairness evaluation with AIF360
# Sensitive attribute: Sex
# =========================================

import numpy as np
import pandas as pd
from aif360.datasets import BinaryLabelDataset
from aif360.metrics import ClassificationMetric

# ------------------------------------------------
# 1. Select the test subset and encode labels/sex
# ------------------------------------------------

# Indices of the test set from Step 1
test_index = X_test.index

# Subset of the original df corresponding to the test set
test_df = df.loc[test_index, ['Sex', 'Risk']].copy()

# Map labels: 'good'/'bad' -> 0/1  (0 = good, 1 = bad)
label_map = {'good': 0, 'bad': 1}
test_df['Risk_num'] = test_df['Risk'].map(label_map)

# Map Sex to numeric (adjust these if your values differ)
# For example, if your data uses 'male'/'female':
sex_map = {'male': 1, 'female': 0}
test_df['Sex_num'] = test_df['Sex'].map(sex_map)

# Safety check: if mapping fails, inspect unique values
if test_df['Sex_num'].isna().any():
    print("WARNING: Some Sex values were not mapped. Check sex_map and df['Sex'].unique().")
    print("Unique Sex values:", test_df['Sex'].unique())

# Encode predictions to numeric using the same label_map
y_true_num = test_df['Risk_num'].values
y_pred_log_num = pd.Series(y_pred_log, index=test_index).map(label_map).values
y_pred_svm_num = pd.Series(y_pred_svm, index=test_index).map(label_map).values

# ------------------------------------------------
# 2. Create BinaryLabelDataset objects
# ------------------------------------------------

# Features for fairness evaluation (only Sex_num is needed here)
X_fair = test_df[['Sex_num']].values

# True dataset
dataset_true = BinaryLabelDataset(
    favorable_label=0,          # 0 = 'good'
    unfavorable_label=1,        # 1 = 'bad'
    df=pd.DataFrame(
        data=np.hstack([X_fair, y_true_num.reshape(-1, 1)]),
        columns=['Sex_num', 'Risk_num']
    ),
    label_names=['Risk_num'],
    protected_attribute_names=['Sex_num']
)

# Logistic regression predictions
dataset_pred_log = dataset_true.copy()
dataset_pred_log.labels = y_pred_log_num.reshape(-1, 1)

# Linear SVM predictions
dataset_pred_svm = dataset_true.copy()
dataset_pred_svm.labels = y_pred_svm_num.reshape(-1, 1)

# ------------------------------------------------
# 3. Define privileged / unprivileged groups
# ------------------------------------------------

# Privileged: Sex_num = 1 (male), Unprivileged: Sex_num = 0 (female)
# Change these if you choose a different mapping above.
privileged_groups = [{'Sex_num': 1}]
unprivileged_groups = [{'Sex_num': 0}]

# ------------------------------------------------
# 4. Helper function to compute fairness metrics
# ------------------------------------------------

def print_fairness_metrics(model_name, dataset_true, dataset_pred):
    metric = ClassificationMetric(
        dataset_true,
        dataset_pred,
        unprivileged_groups=unprivileged_groups,
        privileged_groups=privileged_groups
    )

    print(f"\n=== Fairness metrics for {model_name} ===")
    # Demographic / statistical parity
    spd = metric.statistical_parity_difference()
    di = metric.disparate_impact()

    # Equality of opportunity (TPR difference)
    eod = metric.equal_opportunity_difference()

    # Average odds difference (avg of FPR and TPR diffs)
    aod = metric.average_odds_difference()

    # Inequality measure
    theil = metric.theil_index()

    print(f"Statistical Parity Difference (SPD): {spd:.4f}")
    print(f"Disparate Impact (DI):              {di:.4f}")
    print(f"Equal Opportunity Difference:       {eod:.4f}")
    print(f"Average Odds Difference:            {aod:.4f}")
    print(f"Theil Index:                        {theil:.4f}")
    print("=========================================")

# ------------------------------------------------
# 5. Evaluate both models
# ------------------------------------------------

print_fairness_metrics("Logistic Regression", dataset_true, dataset_pred_log)
print_fairness_metrics("Linear SVM", dataset_true, dataset_pred_svm)


=== Fairness metrics for Logistic Regression ===
Statistical Parity Difference (SPD): -0.0048
Disparate Impact (DI):              0.9949
Equal Opportunity Difference:       0.0116
Average Odds Difference:            -0.0050
Theil Index:                        0.0865

=== Fairness metrics for Linear SVM ===
Statistical Parity Difference (SPD): -0.0048
Disparate Impact (DI):              0.9949
Equal Opportunity Difference:       0.0116
Average Odds Difference:            -0.0050
Theil Index:                        0.0865
